# Setup Folders

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists("src"):
        !git clone https://github.com/HenriqueSchmitz/world-models-implementation temp_repo
        !mv temp_repo/* .
        !rm -rf temp_repo
        !sudo apt-get install -y swig
        %pip install -q -r requirements.txt

In [2]:
import os
from src.utils.colab import is_environment_colab_notebook
from src.utils.secrets import get_secret

settings_path = "./settings.json"
data_folder = "./data"
os.makedirs(data_folder, exist_ok=True)

# Settings

In [3]:
from src.utils.logging import get_logger
LOG_LEVEL = "INFO"
logger = get_logger(LOG_LEVEL)

2026-04-16 11:17:05 [INFO] Logger initialized.


In [4]:
import json

with open(settings_path, "r") as settings_file:
    settings = json.load(settings_file)
    OBSERVATION_CROP_DIM = settings["data_ingestion"]["observation_crop_dim"]
    OBSERVATION_DIM = settings["vae"]["model"]["observation_dim"]
    NUM_EXPLORATION_PROCESSES = settings["data_ingestion"]["num_exploration_processes"]
    TARGET_RUN_COUNT = settings["data_ingestion"]["target_run_count"]
    
def log_settings():
    logger.info(f"OBSERVATION_CROP_DIM: {OBSERVATION_CROP_DIM}")
    logger.info(f"OBSERVATION_DIM: {OBSERVATION_DIM}")
    logger.info(f"NUM_EXPLORATION_PROCESSES: {NUM_EXPLORATION_PROCESSES}")
    logger.info(f"TARGET_RUN_COUNT: {TARGET_RUN_COUNT}")

log_settings()

2026-04-16 11:17:06 [INFO] OBSERVATION_CROP_DIM: 83
2026-04-16 11:17:06 [INFO] OBSERVATION_DIM: 64
2026-04-16 11:17:06 [INFO] NUM_EXPLORATION_PROCESSES: 8
2026-04-16 11:17:06 [INFO] TARGET_RUN_COUNT: 1500


# Install and Load Packages

In [5]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [28 lines of output]
      Using setuptools (version 75.1.0).
      e:\conda_data\envs\learning\lib\site-packages\setuptools\_distutils\dist.py:261: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-38\Box2D
      copying library\Box2D\Box2D.py -> build\lib.win-amd64-cpython-38\Box2D
      copying library\Box2D\__init__.py -> build\lib.win-amd64-cpython-38\Box2D
      creating build\lib.win-amd64-cpython-38\Box2D\b2
      copying library\Box2D\b2\__init__.py -> build\lib.win-amd64-cpython-38\Box2D\b2
      running build_ext
      building 'Box2D._Box2D' extension
      swigging Box2D\Box2D.i to Box2D\Box2D_wrap.cpp
      swig.exe -python -c++ -IBox2D -small -O -includeall -ignoremissing -w201 -globals b2Globals -outdir libra

In [6]:
import itertools
import json
import multiprocessing
import os
from typing import Dict, Any

from tqdm.notebook import tqdm

from src.car_racing_worker import run_single_exploration

In [7]:
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning" # Pygame has a deprecation warning at the moment

# Setup

In [8]:
def ensure_count_file_exists(data_folder):
  run_count_file_path = os.path.join(data_folder, "run_count.txt")
  if not os.path.exists(run_count_file_path):
    with open(run_count_file_path, 'w') as count_file:
      count = str(0).zfill(7)
      count_file.write(str(count))


In [9]:
ensure_count_file_exists(data_folder)

# Functions

In [10]:
def get_current_count(data_folder):
  run_count_file_path = os.path.join(data_folder, "run_count.txt")
  try:
    with open(run_count_file_path, "r") as count_file:
      count_str = count_file.read().strip()
      if count_str:
        return int(count_str)
      return 0
  except (FileNotFoundError, ValueError):
    return 0

In [16]:
def load_metadata(data_folder: str) -> Dict[str, Dict[str, Any]]:
    metadata = {}
    metadata_path = os.path.join(data_folder, "metadata.json")
    if os.path.exists(metadata_path):
        with open(metadata_path, "r") as file:
            metadata = json.load(file)
    return metadata

def save_metadata(metadata: Dict[str, Dict[str, Any]], data_folder: str) -> None:
    metadata_path = os.path.join(data_folder, "metadata.json")
    with open(metadata_path, "w") as file:
        json.dump(metadata, file)

In [17]:
def run_multiple_explorations(data_folder, num_processes, target_count, logger):
    logger.info(f"Starting {num_processes} parallel processes")
    start_count = get_current_count(data_folder)
    if start_count > target_count:
        logger.warning(f"Current next exploration ({start_count}) already meets or exceeds target ({target_count})")
        return
    
    metadata = load_metadata(data_folder)
    ctx = multiprocessing.get_context('spawn')
    with ctx.Pool(processes=num_processes) as pool:
        kwargs = {
            "data_folder": data_folder,
            "y_crop_dim": OBSERVATION_CROP_DIM,
            "observation_dim": OBSERVATION_DIM,
            "logger": logger
        }
        tasks_iterable = itertools.repeat(kwargs)
        with tqdm(total=target_count, desc="Explorations Completed") as pbar:
            pbar.n = start_count
            pbar.refresh()
            result_pool = pool.imap_unordered(run_single_exploration, tasks_iterable)
            for result_count, result_file_name, result_length in result_pool:
                logger.debug(result_count)
                logger.debug(result_file_name)
                logger.debug(result_length)
                metadata[result_file_name] = {"length": result_length}
                if result_count >= target_count:
                    logger.debug(f"\nReached target count {result_count}")
                    logger.debug("Terminating all worker processes...")
                    pool.terminate()
                    pool.join()
                    logger.info(f"Finished all {target_count} explorations")
                    logger.debug(metadata)
                    save_metadata(metadata, data_folder)
                    return
                pbar.n = result_count + 1
                pbar.refresh()

# Run

In [18]:
run_multiple_explorations(data_folder,
                          num_processes=NUM_EXPLORATION_PROCESSES,
                          target_count=TARGET_RUN_COUNT,
                          logger=logger)

2026-04-16 11:34:25 [INFO] Starting 8 parallel processes


Explorations Completed:   0%|          | 0/1500 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Upload

In [14]:
if is_environment_colab_notebook():
    remote_folder = get_secret("worldModelsFolderPath")
    !tar -cf data.tar ./data
    !rm -rf ./data
    !cp ./data.tar "$remote_folder"